In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_data.csv")

# Target Label if it is >0.25 then 1 otherwise it will be 0
df['churn'] = (df['churn_probability'] > 0.25).astype(int)

# dropping columns that aren't needed
df = df.drop(['customer_id', 'churn_probability'], axis=1)

# one hot encoding to convert text based categories into numbers
df = pd.get_dummies(df, drop_first=True)

# Split -> X(features) , Y(target)
X = df.drop('churn', axis=1)
y = df['churn']

# handling missing values, mean imputation
X = X.fillna(X.mean())

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reduce to 4 features (VERY IMPORTANT)
pca = PCA(n_components=4)
X_reduced = pca.fit_transform(X_scaled)

# Small subset
X_small = X_reduced[:200]
y_small = y[:200]

In [3]:
# Convert to numpy
X_small = np.array(X_small)
y_small = np.array(y_small)

# Convert labels (IMPORTANT)
y_small = np.where(y_small == 0, -1, 1)

In [4]:
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit import QuantumCircuit

num_features = X_small.shape[1]

feature_map = ZZFeatureMap(feature_dimension=num_features)
ansatz = RealAmplitudes(num_qubits=num_features)

qc = QuantumCircuit(num_features)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

C:\Users\Shubham\AppData\Local\Temp\ipykernel_12300\1849683334.py:6: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_features)
C:\Users\Shubham\AppData\Local\Temp\ipykernel_12300\1849683334.py:7: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  ansatz = RealAmplitudes(num_qubits=num_features)


In [5]:
from qiskit_machine_learning.neural_networks import EstimatorQNN

qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [6]:
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit.algorithms.optimizers import COBYLA

classifier = NeuralNetworkClassifier(
    qnn,
    optimizer=COBYLA(maxiter=50)
)

ModuleNotFoundError: No module named 'qiskit.algorithms'

# Assistant
This error occurs because the import statement is trying to access a module that doesn't exist or has been moved in the version of Qiskit you're using.

The error specifically mentions that there's no module named 'qiskit.algorithms', which suggests that the structure of the Qiskit library has changed in your installed version.

Would you like me to provide the corrected code?

# User
yes

In [7]:
# Assistant
# The structure of Qiskit has changed in newer versions
# Import COBYLA from qiskit.algorithms.optimizers is now from qiskit_algorithms.optimizers
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import COBYLA  # Updated import path

classifier = NeuralNetworkClassifier(
    qnn,
    optimizer=COBYLA(maxiter=50)
)

ModuleNotFoundError: No module named 'qiskit_algorithms'

In [ ]:
# Assistant
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_algorithms.optimizers import COBYLA  # Changed from qiskit.algorithms to qiskit_algorithms

classifier = NeuralNetworkClassifier(
    qnn,
    optimizer=COBYLA(maxiter=50)
)

In [ ]:
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

classifier = NeuralNetworkClassifier(
    qnn,
    optimizer=COBYLA(maxiter=50)
)

In [ ]:
classifier.fit(X_small, y_small)

In [ ]:
y_pred_qml = classifier.predict(X_small)

In [ ]:
import numpy as np

y_pred_qml = np.where(y_pred_qml == -1, 0, 1)
y_true_qml = np.where(y_small == -1, 0, 1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true_qml, y_pred_qml))

In [ ]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_true_qml, y_pred_qml))